In [5]:
import pandas as pd
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text
from datetime import datetime
from dateutil.relativedelta import relativedelta

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

class Updater:
    def __init__(self):
        
        self.__get_engine()
        
    def __load_dotenv(self):
        load_dotenv()
        return {
            'user': os.getenv("DB_USER"),
            'password': os.getenv("DB_PASSWORD"),
            'host': '172.16.3.158',
            'port': os.getenv("DB_PORT"),
            'db_name': os.getenv("DB_NAME"),
            'schema': 'fin'
        }

    def __get_engine(self):
        conf = self.__load_dotenv()
        self.__engine = create_engine(f"postgresql+psycopg2://{conf['user']}:{conf['password']}@{conf['host']}:{conf['port']}/{conf['db_name']}")

    def read_sql_query(self, query):
        with self.__engine.begin() as con:
            return pd.read_sql_query(query, con)

    def pandas_to_db(self, df, table, exists):
        with self.__engine.connect() as con:
            df.to_sql(table, con, if_exists=exists, index=False, schema='fin')

In [7]:
upd = Updater()

In [16]:
query = """select distinct frc, 'АО "РТ-Техприемка"' as company  from fin.frc_index where rev_frc is true"""
upd.read_sql_query(query)

,frc,company
0,Стратегия и инвестиции,"АО ""РТ-Техприемка"""
1,Центр качества поставок,"АО ""РТ-Техприемка"""
2,Метрологическая служба,"АО ""РТ-Техприемка"""
3,"Система добровольной сертификации ""Ростех""","АО ""РТ-Техприемка"""
4,Оценка и технический контроль,"АО ""РТ-Техприемка"""
5,Центр компетенции и системы управления качеством ГК Ростех,"АО ""РТ-Техприемка"""
6,Инновации и инжиниринг,"АО ""РТ-Техприемка"""
7,Управление проектами и цифровизацией,"АО ""РТ-Техприемка"""
8,Прочие,"АО ""РТ-Техприемка"""
9,Направление продаж,"АО ""РТ-Техприемка"""


In [17]:
class YearUpdater:
    def __init__(self):
        self.__get_engine()

    def __load_dotenv(self):
        load_dotenv()
        return {
            'user': os.getenv("DB_USER"),
            'password': os.getenv("DB_PASSWORD"),
            'host': os.getenv("DB_HOST"),
            'port': os.getenv("DB_PORT"),
            'db_name': os.getenv("DB_NAME"),
            'schema': 'fin'
        }

    def __get_engine(self):
        conf = self.__load_dotenv()
        self.__engine = create_engine(f"postgresql+psycopg2://{conf['user']}:{conf['password']}@{conf['host']}:{conf['port']}/{conf['db_name']}")

    def __read_sql_query(self, query):
        with self.__engine.begin() as con:
            return pd.read_sql_query(query, con)

    def __get_all_frc(self):
        c_year = datetime.now().year
        query = f"""
            select distinct frc, 'АО "РТ-Техприемка"' as company  
            from fin.frc_index 
            where rev_frc is true
            """
        return self.__read_sql_query(query)


    def __get_all_dates(self):
        c_year = datetime.now().year
        return pd.DataFrame({'date_dt': [datetime(c_year, 1, 1).date() + relativedelta(months=i) for i in range(12)],
                            'estimate_date': [datetime(c_year, 1, 1).date()] * 12,
                            'est_amount': [None] * 12,
                            'hcl_amount': [None] * 12,
                            'contr_amount': [None] * 12})

    def update_table(self):
        company_frc = self.__get_all_frc()
        dates_amounts = self.__get_all_dates()
        result_df = company_frc.join(dates_amounts, how='cross')[['company', 'date_dt',
                                                                  'estimate_date', 'frc',
                                                                 'est_amount', 'hcl_amount', 'contr_amount']]
        return result_df
        # with self.__engine.connect() as con:
        #     result_df.to_sql("revenue_est_2025", con, schema='fin', if_exists='append', index=False)

In [18]:
yu = YearUpdater()
yu.update_table()

,company,date_dt,estimate_date,frc,est_amount,hcl_amount,contr_amount
0,"АО ""РТ-Техприемка""",2025-01-01,2025-01-01,Стратегия и инвестиции,None,None,None
1,"АО ""РТ-Техприемка""",2025-02-01,2025-01-01,Стратегия и инвестиции,None,None,None
2,"АО ""РТ-Техприемка""",2025-03-01,2025-01-01,Стратегия и инвестиции,None,None,None
3,"АО ""РТ-Техприемка""",2025-04-01,2025-01-01,Стратегия и инвестиции,None,None,None
4,"АО ""РТ-Техприемка""",2025-05-01,2025-01-01,Стратегия и инвестиции,None,None,None
5,"АО ""РТ-Техприемка""",2025-06-01,2025-01-01,Стратегия и инвестиции,None,None,None
6,"АО ""РТ-Техприемка""",2025-07-01,2025-01-01,Стратегия и инвестиции,None,None,None
7,"АО ""РТ-Техприемка""",2025-08-01,2025-01-01,Стратегия и инвестиции,None,None,None
8,"АО ""РТ-Техприемка""",2025-09-01,2025-01-01,Стратегия и инвестиции,None,None,None
9,"АО ""РТ-Техприемка""",2025-10-01,2025-01-01,Стратегия и инвестиции,None,None,None
